# release16_wf2020_trend_participation_gap_v1

## Governance

- Baseline: `release16_downtrend_drawdown_objective_screen_v1` selected `wf_2020/dd_control_checkpoint` official warm-start daily path.
- Direct comparison baseline: `soxlbacktest_qlib_walkforward_baseline_v1` `legacy_default`, plus its drawdown-safe risk probes.
- Role: diagnose. This card does not retain an actor and does not promote a runtime overlay.
- Target window: `wf_2020` first, because release16 already repaired all three folds below the user requested `<20%` MaxDD threshold but misses the `+30pp` relative-return floor in `wf_2020`.
- Changed layer: model-selection / target-design diagnostics, not production execution.
- External reference: existing SOXLBacktest qlib buy/sell reference and repo-owned `TrendParticipationGuardConfig` feature semantics.
- Hypothesis: remaining `wf_2020` gap is caused by under-participation in high-trend low-risk states; a simple runtime trend floor is likely insufficient or leakage-prone, but the gap can be turned into a fold-local training/checkpoint target if the state is visible in train-safe features.
- Acceptance for this diagnostic: identify the dated return-gap drivers, compare against SOXLBacktest buy/sell timing, and screen non-promotable counterfactual trend-participation floors under both action-aligned and lagged semantics.
- Failure mode: if only same-day/action-aligned counterfactuals pass, do not promote; convert the signal to fold-local target/checkpoint objective and rerun official replay.
- Historical success case: `release16_downtrend_drawdown_objective_screen_v1` got all official folds under `<20%` MaxDD.
- Historical failure case: `wf2020_*floor*` and trend floor screens repeatedly improved or exposed participation but did not produce a promotable official three-fold actor.
- Duplicate guard: this card starts from the latest release16 official daily path, compares to the qlib SOXLBacktest baseline, and explicitly checks lagged versus same-day semantics before recommending another training target.


In [1]:
from __future__ import annotations

import itertools
import json
import math
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "SPEC.md").exists():
            return candidate
    raise RuntimeError(f"could not find repo root from {start}")

ROOT = find_repo_root(Path.cwd().resolve())
CARD = ROOT / "runtime" / "model_selection" / "card" / "release16_wf2020_trend_participation_gap_v1"
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_CARD = ROOT / "runtime" / "model_selection" / "card" / "release16_downtrend_drawdown_objective_screen_v1"
SOXLBT_CARD = ROOT / "runtime" / "model_selection" / "card" / "soxlbacktest_qlib_walkforward_baseline_v1"

ARTIFACTS = {
    "gap_contribution": CARD / "wf2020_gap_contribution.csv",
    "monthly_gap": CARD / "wf2020_monthly_gap_contribution.csv",
    "drawdown_intervals": CARD / "wf2020_drawdown_intervals.csv",
    "trend_overlay_grid": CARD / "trend_overlay_grid.csv",
    "trend_overlay_top_daily": CARD / "trend_overlay_top_daily.csv",
    "feature_state_summary": CARD / "feature_state_summary.csv",
    "selection": CARD / "selection.json",
    "summary": CARD / "summary.json",
    "conclusion": CARD / "conclusion.md",
}
ARTIFACTS


{'gap_contribution': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_wf2020_trend_participation_gap_v1/wf2020_gap_contribution.csv'),
 'monthly_gap': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_wf2020_trend_participation_gap_v1/wf2020_monthly_gap_contribution.csv'),
 'drawdown_intervals': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_wf2020_trend_participation_gap_v1/wf2020_drawdown_intervals.csv'),
 'trend_overlay_grid': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_wf2020_trend_participation_gap_v1/trend_overlay_grid.csv'),
 'trend_overlay_top_daily': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_wf2020_trend_p

In [2]:
def pct_return(values: pd.Series) -> float:
    vals = pd.to_numeric(values, errors="coerce").dropna()
    if len(vals) < 2:
        return 0.0
    return float((vals.iloc[-1] / vals.iloc[0] - 1.0) * 100.0)


def max_drawdown(values: pd.Series) -> float:
    vals = pd.to_numeric(values, errors="coerce").dropna().astype(float)
    if len(vals) < 2:
        return 0.0
    peak = vals.cummax()
    dd = vals / peak - 1.0
    return abs(float(dd.min()) * 100.0)


def max_drawdown_interval(frame: pd.DataFrame, nav_col: str = "strategy_nav") -> dict[str, Any]:
    data = frame.copy()
    data["datetime"] = pd.to_datetime(data["datetime"])
    data = data.sort_values("datetime").reset_index(drop=True)
    nav = pd.to_numeric(data[nav_col], errors="coerce").ffill()
    peak = nav.cummax()
    dd = nav / peak - 1.0
    trough_idx = int(dd.idxmin())
    peak_idx = int(nav.loc[:trough_idx].idxmax())
    interval = data.loc[peak_idx:trough_idx]
    pos = pd.to_numeric(interval.get("position", pd.Series(0.0, index=interval.index)), errors="coerce").fillna(0.0)
    return {
        "peak_date": str(pd.Timestamp(data.loc[peak_idx, "datetime"]).date()),
        "trough_date": str(pd.Timestamp(data.loc[trough_idx, "datetime"]).date()),
        "max_dd_pct": abs(float(dd.iloc[trough_idx]) * 100.0),
        "interval_days": int(len(interval)),
        "interval_avg_pos": float(pos.mean()) if len(pos) else 0.0,
        "interval_min_pos": float(pos.min()) if len(pos) else 0.0,
        "interval_max_pos": float(pos.max()) if len(pos) else 0.0,
    }


def path_metrics(frame: pd.DataFrame, label: str) -> dict[str, Any]:
    data = frame.copy()
    data["datetime"] = pd.to_datetime(data["datetime"])
    data = data.sort_values("datetime")
    strategy_return = pct_return(data["strategy_nav"])
    buyhold_return = pct_return(data["buyhold_nav"])
    return {
        "label": label,
        "rows": int(len(data)),
        "start": str(data["datetime"].iloc[0].date()),
        "end": str(data["datetime"].iloc[-1].date()),
        "strategy_return_pct": strategy_return,
        "buyhold_return_pct": buyhold_return,
        "relative_return_pp": strategy_return - buyhold_return,
        "max_dd_pct": max_drawdown(data["strategy_nav"]),
        "avg_pos": float(pd.to_numeric(data["position"], errors="coerce").mean()),
        "trade_count": int(data.get("action", pd.Series(index=data.index, dtype=object)).astype(str).str.upper().isin(["BUY", "SELL"]).sum()),
    }


def rolling_percentile(series: pd.Series, window: int) -> pd.Series:
    def pct(x: np.ndarray) -> float:
        if len(x) == 0 or np.isnan(x[-1]):
            return np.nan
        valid = x[~np.isnan(x)]
        if len(valid) == 0:
            return np.nan
        return float((valid <= x[-1]).mean())
    return series.rolling(window, min_periods=max(20, window // 4)).apply(pct, raw=True)


def build_feature_frame(panel: pd.DataFrame) -> pd.DataFrame:
    f = panel.copy()
    f["datetime"] = pd.to_datetime(f["date"])
    f = f.sort_values("datetime").reset_index(drop=True)
    close = pd.to_numeric(f["soxl_close"], errors="coerce")
    soxx = pd.to_numeric(f["soxx_close"], errors="coerce")
    vix = pd.to_numeric(f["vix"], errors="coerce")
    ret = close.pct_change()
    soxx_ret = soxx.pct_change()
    ma21 = close.rolling(21, min_periods=10).mean()
    ma63 = close.rolling(63, min_periods=21).mean()
    ma180 = close.rolling(180, min_periods=60).mean()
    out = pd.DataFrame({"datetime": f["datetime"]})
    out["mom_21"] = ret.rolling(21, min_periods=10).sum()
    out["mom_63"] = ret.rolling(63, min_periods=21).sum()
    out["relative_strength_21"] = (ret - soxx_ret).rolling(21, min_periods=10).sum()
    out["trend_gap_63"] = ma21 / ma63 - 1.0
    out["anchor_gap_180"] = close / ma180 - 1.0
    out["trend_slope_63"] = ma63 / ma63.shift(21) - 1.0
    out["realized_vol_63"] = ret.rolling(63, min_periods=21).std() * math.sqrt(252)
    out["drawdown_63"] = close / close.rolling(63, min_periods=21).max() - 1.0
    out["vix_percentile_252"] = rolling_percentile(vix, 252)
    return out


def replay_nav(frame: pd.DataFrame, positions: pd.Series, mode: str) -> pd.Series:
    data = frame.copy().sort_values("datetime").reset_index(drop=True)
    price = pd.to_numeric(data["price"], errors="coerce").ffill().bfill().reset_index(drop=True)
    pos = pd.to_numeric(positions, errors="coerce").fillna(0.0).clip(0.0, 1.0).reset_index(drop=True)
    nav = [float(pd.to_numeric(data["strategy_nav"], errors="coerce").iloc[0])]
    for i in range(1, len(data)):
        if mode == "same_day":
            exposure = float(pos.iloc[i])
        elif mode == "lagged":
            exposure = float(pos.iloc[i - 1])
        else:
            raise ValueError(mode)
        nav.append(nav[-1] * (1.0 + exposure * (float(price.iloc[i] / price.iloc[i - 1]) - 1.0)))
    return pd.Series(nav, index=data.index)


def evaluate_positions(frame: pd.DataFrame, positions: pd.Series, label: str, mode: str) -> tuple[dict[str, Any], pd.DataFrame]:
    out = frame.copy().sort_values("datetime").reset_index(drop=True)
    out["base_position"] = pd.to_numeric(out["position"], errors="coerce").clip(0.0, 1.0)
    out["position"] = pd.to_numeric(positions, errors="coerce").fillna(out["base_position"]).clip(0.0, 1.0)
    out["base_strategy_nav"] = pd.to_numeric(out["strategy_nav"], errors="coerce")
    out["strategy_nav"] = replay_nav(out, out["position"], mode=mode)
    metrics = path_metrics(out, label)
    metrics["mode"] = mode
    metrics["lifted_days"] = int((out["position"] > out["base_position"] + 1e-12).sum())
    metrics["avg_lift"] = float((out["position"] - out["base_position"]).clip(lower=0.0).mean())
    metrics["max_lift"] = float((out["position"] - out["base_position"]).clip(lower=0.0).max())
    return metrics, out


In [3]:
selected = pd.read_csv(
    SOURCE_CARD / "official_daily_paths" / "wf_2020__dd_control_checkpoint_official_daily_path.csv",
    parse_dates=["datetime"],
).sort_values("datetime")
release_metrics = pd.read_csv(SOURCE_CARD / "bc_candidate_metrics.csv")
soxlbt_metrics = pd.read_csv(SOXLBT_CARD / "candidate_metrics.csv")
soxlbt_daily = pd.read_csv(SOXLBT_CARD / "candidate_daily.csv", parse_dates=["date"])
panel = pd.read_csv(ROOT / "data" / "processed" / "panels" / "soxl_soxx_vix_full.csv")
features = build_feature_frame(panel)

selected_metrics = path_metrics(selected, "release16_dd_control_checkpoint")
selected_metrics


{'label': 'release16_dd_control_checkpoint',
 'rows': 252,
 'start': '2020-01-02',
 'end': '2020-12-30',
 'strategy_return_pct': 52.49076200614864,
 'buyhold_return_pct': 58.093570616483994,
 'relative_return_pp': -5.602808610335352,
 'max_dd_pct': 16.632463415783693,
 'avg_pos': 0.2611765782775197,
 'trade_count': 40}

In [4]:
# Align SOXLBacktest references onto the release16 official dates.
refs = []
for candidate in ["legacy_default", "risk_cap_35_15_05", "wf_pass_ytd18_15_15_02", "low_risk_20_08_02"]:
    sub = soxlbt_daily[(soxlbt_daily["candidate"].eq(candidate)) & (soxlbt_daily["window"].eq("wf_2020"))].copy()
    sub = sub.rename(columns={"date": "datetime"})
    sub["datetime"] = pd.to_datetime(sub["datetime"])
    refs.append(sub[["datetime", "candidate", "position", "strategy_nav", "buyhold_nav", "action", "price"]])

comparison_rows = [selected_metrics]
for ref in refs:
    name = str(ref["candidate"].iloc[0])
    aligned = selected[["datetime"]].merge(ref, on="datetime", how="left")
    # SOXLBacktest starts with 100k NAV; metrics remain self-contained.
    comparison_rows.append(path_metrics(aligned.dropna(subset=["strategy_nav", "buyhold_nav"]), f"soxlbacktest_{name}"))
comparison = pd.DataFrame(comparison_rows)
comparison


,label,rows,start,end,strategy_return_pct,buyhold_return_pct,relative_return_pp,max_dd_pct,avg_pos,trade_count
0,release16_dd_control_checkpoint,252,2020-01-02,2020-12-30,52.490762,58.093571,-5.602809,16.632463,0.261177,40
1,soxlbacktest_legacy_default,252,2020-01-02,2020-12-30,217.458044,58.093571,159.364474,54.275144,0.790268,4
2,soxlbacktest_risk_cap_35_15_05,252,2020-01-02,2020-12-30,68.568869,58.093571,10.475298,17.942350,0.309346,4
3,soxlbacktest_wf_pass_ytd18_15_15_02,252,2020-01-02,2020-12-30,28.672477,58.093571,-29.421094,12.785207,0.170365,2
4,soxlbacktest_low_risk_20_08_02,252,2020-01-02,2020-12-30,35.068955,58.093571,-23.024616,9.073791,0.173237,5


In [5]:
base = selected[["datetime", "price", "strategy_nav", "buyhold_nav", "position", "action"]].copy()
base = base.rename(columns={"position": "release16_position", "strategy_nav": "release16_nav"})
for ref in refs:
    name = str(ref["candidate"].iloc[0])
    cols = ref[["datetime", "position", "action"]].rename(
        columns={"position": f"{name}_position", "action": f"{name}_action"}
    )
    base = base.merge(cols, on="datetime", how="left")
base = base.merge(features, on="datetime", how="left")
base["price_return"] = pd.to_numeric(base["price"], errors="coerce").pct_change().fillna(0.0)

for teacher in ["legacy_default", "risk_cap_35_15_05", "wf_pass_ytd18_15_15_02", "low_risk_20_08_02"]:
    gap = pd.to_numeric(base[f"{teacher}_position"], errors="coerce").fillna(0.0) - pd.to_numeric(base["release16_position"], errors="coerce").fillna(0.0)
    base[f"{teacher}_position_gap"] = gap
    base[f"{teacher}_missed_upside_pp"] = (gap.clip(lower=0.0) * base["price_return"].clip(lower=0.0) * 100.0)
    base[f"{teacher}_net_gap_contribution_pp"] = gap * base["price_return"] * 100.0

rank_cols = [
    "datetime", "price", "price_return", "release16_position", "legacy_default_position", "risk_cap_35_15_05_position",
    "legacy_default_position_gap", "legacy_default_missed_upside_pp", "legacy_default_net_gap_contribution_pp",
    "anchor_gap_180", "mom_63", "trend_slope_63", "trend_gap_63", "vix_percentile_252", "realized_vol_63", "drawdown_63",
]
gap_contribution = base[rank_cols].sort_values("legacy_default_missed_upside_pp", ascending=False)
gap_contribution.to_csv(ARTIFACTS["gap_contribution"], index=False)
gap_contribution.head(15)


,datetime,price,price_return,release16_position,legacy_default_position,risk_cap_35_15_05_position,legacy_default_position_gap,legacy_default_missed_upside_pp,legacy_default_net_gap_contribution_pp,anchor_gap_180,mom_63,trend_slope_63,trend_gap_63,vix_percentile_252,realized_vol_63,drawdown_63
65,2020-04-06,7.102667,0.306758,0.125000,0.9192,0.2131,0.794200,24.362754,24.362754,-0.475455,-0.353825,-0.199839,-0.541584,0.920635,2.083786,-0.673170
56,2020-03-24,6.249333,0.341059,0.250585,0.9417,0.1912,0.691115,23.571078,23.571078,-0.545895,-0.612742,-0.103855,-0.381047,0.976190,1.869296,-0.712436
49,2020-03-13,8.661333,0.273226,0.088337,0.9417,0.1737,0.853363,23.316105,23.316105,-0.379396,-0.331337,0.003158,-0.145400,0.996032,1.438954,-0.601448
58,2020-03-26,7.205333,0.199956,0.250000,0.9417,0.1912,0.691700,13.830925,13.830925,-0.474986,-0.465058,-0.126145,-0.415436,0.964286,1.916487,-0.668446
76,2020-04-22,8.039333,0.167490,0.142862,0.9192,0.2131,0.776338,13.002855,13.002855,-0.395252,-0.333806,-0.245043,-0.428802,0.900794,2.168428,-0.630069
81,2020-04-29,9.760000,0.143482,0.125000,0.9418,0.2781,0.816800,11.719619,11.719619,-0.264276,-0.016728,-0.253388,-0.341042,0.829365,2.173706,-0.550893
70,2020-04-14,8.462000,0.132798,0.145164,0.9192,0.2131,0.774036,10.279035,10.279035,-0.368580,-0.252744,-0.231429,-0.542731,0.884921,2.112350,-0.610620
94,2020-05-18,9.232667,0.139929,0.223988,0.9418,0.2781,0.717812,10.044298,10.044298,-0.298569,-0.147031,-0.285967,-0.093806,0.785714,2.216737,-0.575158
154,2020-08-12,17.070000,0.100722,0.104889,0.9645,0.3930,0.859611,8.658183,8.658183,0.288983,0.862044,0.214374,0.182187,0.535714,0.894501,0.000000
46,2020-03-10,11.773333,0.190910,0.116382,0.5494,0.1703,0.433018,8.266728,8.266728,-0.157026,-0.070950,0.035535,-0.066601,0.996032,1.140214,-0.458249


In [6]:
monthly = base.copy()
monthly["month"] = monthly["datetime"].dt.to_period("M").astype(str)
monthly_rows = []
for teacher in ["legacy_default", "risk_cap_35_15_05", "wf_pass_ytd18_15_15_02", "low_risk_20_08_02"]:
    agg = monthly.groupby("month").agg(
        rows=("datetime", "size"),
        price_return_pct=("price_return", lambda s: float(((1.0 + s).prod() - 1.0) * 100.0)),
        release16_avg_pos=("release16_position", "mean"),
        teacher_avg_pos=(f"{teacher}_position", "mean"),
        missed_upside_pp=(f"{teacher}_missed_upside_pp", "sum"),
        net_gap_contribution_pp=(f"{teacher}_net_gap_contribution_pp", "sum"),
        trend_signal_share=("anchor_gap_180", lambda s: np.nan),
    ).reset_index()
    agg["teacher"] = teacher
    monthly_rows.append(agg)
monthly_gap = pd.concat(monthly_rows, ignore_index=True)
monthly_gap = monthly_gap.sort_values(["teacher", "missed_upside_pp"], ascending=[True, False])
monthly_gap.to_csv(ARTIFACTS["monthly_gap"], index=False)
monthly_gap[monthly_gap["teacher"].eq("legacy_default")].head(12)


,month,rows,price_return_pct,release16_avg_pos,teacher_avg_pos,missed_upside_pp,net_gap_contribution_pp,trend_signal_share,teacher
2,2020-03,22,-53.785367,0.166829,0.734250,99.860499,-22.726179,NaN,legacy_default
3,2020-04,21,38.283727,0.142426,0.923505,92.504792,35.664409,NaN,legacy_default
4,2020-05,20,17.632241,0.190684,0.943060,54.051389,14.933420,NaN,legacy_default
10,2020-11,20,63.072910,0.306913,0.975580,44.264698,28.059191,NaN,legacy_default
5,2020-06,22,22.127045,0.345127,0.951018,40.353015,21.210743,NaN,legacy_default
6,2020-07,22,20.349618,0.264564,0.960873,39.041098,13.955741,NaN,legacy_default
8,2020-09,21,-5.207790,0.370047,0.967881,32.461826,5.076940,NaN,legacy_default
7,2020-08,21,17.919740,0.167032,0.966214,31.429173,14.402794,NaN,legacy_default
9,2020-10,22,-1.674635,0.263985,0.971345,29.095552,1.610671,NaN,legacy_default
11,2020-12,21,13.695594,0.302807,0.979319,22.215909,9.430872,NaN,legacy_default


In [7]:
drawdown_rows = []
for label, frame in [("release16_dd_control_checkpoint", selected)]:
    row = {"label": label, **max_drawdown_interval(frame)}
    drawdown_rows.append(row)
for ref in refs:
    name = str(ref["candidate"].iloc[0])
    aligned = selected[["datetime"]].merge(ref.rename(columns={"date": "datetime"}), on="datetime", how="left")
    aligned = aligned.rename(columns={"position": "position"})
    if aligned["strategy_nav"].notna().sum() > 2:
        drawdown_rows.append({"label": f"soxlbacktest_{name}", **max_drawdown_interval(aligned.dropna(subset=["strategy_nav"]))})

drawdown_intervals = pd.DataFrame(drawdown_rows)
drawdown_intervals.to_csv(ARTIFACTS["drawdown_intervals"], index=False)
drawdown_intervals


,label,peak_date,trough_date,max_dd_pct,interval_days,interval_avg_pos,interval_min_pos,interval_max_pos
0,release16_dd_control_checkpoint,2020-09-02,2020-09-08,16.632463,4,0.537250,0.48132,0.5750
1,soxlbacktest_legacy_default,2020-03-04,2020-03-20,54.275144,13,0.692254,0.31490,0.9417
2,soxlbacktest_risk_cap_35_15_05,2020-03-04,2020-03-20,17.942350,13,0.176277,0.15910,0.1912
3,soxlbacktest_wf_pass_ytd18_15_15_02,2020-03-04,2020-03-20,12.785207,13,0.159100,0.15910,0.1591
4,soxlbacktest_low_risk_20_08_02,2020-03-04,2020-03-20,9.073791,13,0.082638,0.07860,0.0852


In [8]:
feature_base = selected.copy().merge(features, on="datetime", how="left")
feature_base["position"] = pd.to_numeric(feature_base["position"], errors="coerce").clip(0.0, 1.0)

# Reconstruction under unchanged positions: same-day action-aligned semantics matches official final NAV closely;
# lagged semantics is retained as a non-leaking stress view.
recon_rows = []
for mode in ["same_day", "lagged"]:
    m, _ = evaluate_positions(feature_base, feature_base["position"], f"base_reconstruction_{mode}", mode)
    m["final_nav_error_pct"] = (m["strategy_return_pct"] - selected_metrics["strategy_return_pct"])
    m["official_relative_return_pp"] = selected_metrics["relative_return_pp"]
    recon_rows.append(m)
reconstruction = pd.DataFrame(recon_rows)
reconstruction


,label,rows,start,end,strategy_return_pct,buyhold_return_pct,relative_return_pp,max_dd_pct,avg_pos,trade_count,mode,lifted_days,avg_lift,max_lift,final_nav_error_pct,official_relative_return_pp
0,base_reconstruction_same_day,252,2020-01-02,2020-12-30,52.179328,58.093571,-5.914243,17.892327,0.261177,40,same_day,0,0.0,0.0,-0.311434,-5.602809
1,base_reconstruction_lagged,252,2020-01-02,2020-12-30,35.171067,58.093571,-22.922504,16.948866,0.261177,40,lagged,0,0.0,0.0,-17.319695,-5.602809


In [9]:
def signal_mask(frame: pd.DataFrame, cfg: dict[str, Any]) -> pd.Series:
    ready = pd.Series(True, index=frame.index)
    ready &= pd.to_numeric(frame["position"], errors="coerce").fillna(0.0) <= float(cfg["max_base_position"])
    if cfg["min_anchor_gap_180"] is not None:
        ready &= pd.to_numeric(frame["anchor_gap_180"], errors="coerce") >= float(cfg["min_anchor_gap_180"])
    if cfg["min_mom_63"] is not None:
        ready &= pd.to_numeric(frame["mom_63"], errors="coerce") >= float(cfg["min_mom_63"])
    if cfg["min_trend_slope_63"] is not None:
        ready &= pd.to_numeric(frame["trend_slope_63"], errors="coerce") >= float(cfg["min_trend_slope_63"])
    if cfg["min_trend_gap_63"] is not None:
        ready &= pd.to_numeric(frame["trend_gap_63"], errors="coerce") >= float(cfg["min_trend_gap_63"])
    if cfg["max_vix_percentile_252"] is not None:
        ready &= pd.to_numeric(frame["vix_percentile_252"], errors="coerce") <= float(cfg["max_vix_percentile_252"])
    if cfg["max_realized_vol_63"] is not None:
        ready &= pd.to_numeric(frame["realized_vol_63"], errors="coerce") <= float(cfg["max_realized_vol_63"])
    if cfg["min_drawdown_63"] is not None:
        ready &= pd.to_numeric(frame["drawdown_63"], errors="coerce") >= float(cfg["min_drawdown_63"])
    return ready.fillna(False).astype(bool)


def apply_floor_positions(frame: pd.DataFrame, cfg: dict[str, Any]) -> tuple[pd.Series, pd.Series]:
    base_pos = pd.to_numeric(frame["position"], errors="coerce").fillna(0.0).clip(0.0, 1.0)
    ready = signal_mask(frame, cfg)
    lifted = base_pos.where(~ready, np.maximum(base_pos, float(cfg["min_target"]))).clip(upper=float(cfg["max_target"]))
    lifted = np.maximum(base_pos, lifted).clip(0.0, 1.0)
    return pd.Series(lifted, index=frame.index), ready


def overlay_score(cfg: dict[str, Any], mode: str) -> tuple[dict[str, Any], pd.DataFrame]:
    positions, ready = apply_floor_positions(feature_base, cfg)
    if cfg["max_equity_drawdown"] is not None:
        # Apply a simple account-drawdown guard using the replay mode being scored.
        guarded = positions.copy()
        temp_nav = replay_nav(feature_base, guarded, mode=mode)
        dd = 1.0 - temp_nav / temp_nav.cummax()
        blocked = dd > float(cfg["max_equity_drawdown"])
        guarded.loc[blocked] = pd.to_numeric(feature_base.loc[blocked, "position"], errors="coerce").fillna(0.0)
        positions = guarded
    metrics, out = evaluate_positions(feature_base, positions, "trend_floor", mode)
    metrics.update(cfg)
    metrics["signal_days"] = int(ready.sum())
    metrics["gate_pass_20dd"] = bool(metrics["relative_return_pp"] >= 30.0 and metrics["max_dd_pct"] < 20.0)
    metrics["gate_pass_15dd"] = bool(metrics["relative_return_pp"] >= 30.0 and metrics["max_dd_pct"] <= 15.0)
    metrics["objective_loss_20dd"] = max(0.0, 30.0 - float(metrics["relative_return_pp"])) + 4.0 * max(0.0, float(metrics["max_dd_pct"]) - 20.0)
    out["trend_signal_ready"] = ready.values
    out["trend_participation_lifted"] = out["position"] > out["base_position"] + 1e-12
    out["mode"] = mode
    return metrics, out

configs = []
for min_target, max_target in [(0.45, 0.65), (0.55, 0.75), (0.65, 0.85), (0.75, 0.95)]:
    for max_base_position, min_anchor, min_mom, min_slope, min_gap, max_vix, max_dd_guard in itertools.product(
        [0.30, 0.40, 0.55],
        [0.10, 0.25, 0.40, 0.55],
        [0.10, 0.20, 0.35],
        [0.00, 0.05],
        [0.00, 0.05],
        [0.45, 0.60, 0.75, 0.90],
        [0.15, 0.20, None],
    ):
        configs.append({
            "min_target": min_target,
            "max_target": max_target,
            "max_base_position": max_base_position,
            "min_anchor_gap_180": min_anchor,
            "min_mom_63": min_mom,
            "min_trend_slope_63": min_slope,
            "min_trend_gap_63": min_gap,
            "max_vix_percentile_252": max_vix,
            "max_realized_vol_63": 1.00,
            "min_drawdown_63": -0.25,
            "max_equity_drawdown": max_dd_guard,
        })

rows = []
top_daily: list[pd.DataFrame] = []
for cfg in configs:
    for mode in ["same_day", "lagged"]:
        metrics, out = overlay_score(cfg, mode)
        rows.append(metrics)

grid = pd.DataFrame(rows).sort_values(["gate_pass_20dd", "objective_loss_20dd", "relative_return_pp"], ascending=[False, True, False])
grid.to_csv(ARTIFACTS["trend_overlay_grid"], index=False)
grid.head(20)


,label,rows,start,end,strategy_return_pct,buyhold_return_pct,relative_return_pp,max_dd_pct,avg_pos,trade_count,...,min_trend_slope_63,min_trend_gap_63,max_vix_percentile_252,max_realized_vol_63,min_drawdown_63,max_equity_drawdown,signal_days,gate_pass_20dd,gate_pass_15dd,objective_loss_20dd
12876,trend_floor,252,2020-01-02,2020-12-30,207.421268,58.093571,149.327697,17.892327,0.472223,40,...,0.00,0.0,0.75,1.0,-0.25,0.15,104,True,False,0.0
12878,trend_floor,252,2020-01-02,2020-12-30,207.421268,58.093571,149.327697,17.892327,0.472223,40,...,0.00,0.0,0.75,1.0,-0.25,0.20,104,True,False,0.0
12880,trend_floor,252,2020-01-02,2020-12-30,207.421268,58.093571,149.327697,17.892327,0.472223,40,...,0.00,0.0,0.75,1.0,-0.25,NaN,104,True,False,0.0
12924,trend_floor,252,2020-01-02,2020-12-30,207.421268,58.093571,149.327697,17.892327,0.472223,40,...,0.05,0.0,0.75,1.0,-0.25,0.15,104,True,False,0.0
12926,trend_floor,252,2020-01-02,2020-12-30,207.421268,58.093571,149.327697,17.892327,0.472223,40,...,0.05,0.0,0.75,1.0,-0.25,0.20,104,True,False,0.0
12928,trend_floor,252,2020-01-02,2020-12-30,207.421268,58.093571,149.327697,17.892327,0.472223,40,...,0.05,0.0,0.75,1.0,-0.25,NaN,104,True,False,0.0
12882,trend_floor,252,2020-01-02,2020-12-30,194.770253,58.093571,136.676682,17.892327,0.476188,40,...,0.00,0.0,0.90,1.0,-0.25,0.15,107,True,False,0.0
12884,trend_floor,252,2020-01-02,2020-12-30,194.770253,58.093571,136.676682,17.892327,0.476188,40,...,0.00,0.0,0.90,1.0,-0.25,0.20,107,True,False,0.0
12886,trend_floor,252,2020-01-02,2020-12-30,194.770253,58.093571,136.676682,17.892327,0.476188,40,...,0.00,0.0,0.90,1.0,-0.25,NaN,107,True,False,0.0
12930,trend_floor,252,2020-01-02,2020-12-30,194.770253,58.093571,136.676682,17.892327,0.476188,40,...,0.05,0.0,0.90,1.0,-0.25,0.15,107,True,False,0.0


In [10]:
best_by_mode = grid.sort_values(["mode", "objective_loss_20dd", "relative_return_pp"], ascending=[True, True, False]).groupby("mode").head(5)
full_gate = grid[grid["gate_pass_20dd"]]
print("full_gate_count", len(full_gate))
best_by_mode


full_gate_count 3802


,label,rows,start,end,strategy_return_pct,buyhold_return_pct,relative_return_pp,max_dd_pct,avg_pos,trade_count,...,min_trend_slope_63,min_trend_gap_63,max_vix_percentile_252,max_realized_vol_63,min_drawdown_63,max_equity_drawdown,signal_days,gate_pass_20dd,gate_pass_15dd,objective_loss_20dd
10377,trend_floor,252,2020-01-02,2020-12-30,92.489390,58.093571,34.395819,19.072100,0.430626,40,...,0.00,0.0,0.60,1.0,-0.25,0.20,73,True,False,0.0
10379,trend_floor,252,2020-01-02,2020-12-30,92.489390,58.093571,34.395819,19.072100,0.430626,40,...,0.00,0.0,0.60,1.0,-0.25,NaN,73,True,False,0.0
10425,trend_floor,252,2020-01-02,2020-12-30,92.489390,58.093571,34.395819,19.072100,0.430626,40,...,0.05,0.0,0.60,1.0,-0.25,0.20,73,True,False,0.0
10427,trend_floor,252,2020-01-02,2020-12-30,92.489390,58.093571,34.395819,19.072100,0.430626,40,...,0.05,0.0,0.60,1.0,-0.25,NaN,73,True,False,0.0
10473,trend_floor,252,2020-01-02,2020-12-30,92.489390,58.093571,34.395819,19.072100,0.430626,40,...,0.00,0.0,0.60,1.0,-0.25,0.20,73,True,False,0.0
12876,trend_floor,252,2020-01-02,2020-12-30,207.421268,58.093571,149.327697,17.892327,0.472223,40,...,0.00,0.0,0.75,1.0,-0.25,0.15,104,True,False,0.0
12878,trend_floor,252,2020-01-02,2020-12-30,207.421268,58.093571,149.327697,17.892327,0.472223,40,...,0.00,0.0,0.75,1.0,-0.25,0.20,104,True,False,0.0
12880,trend_floor,252,2020-01-02,2020-12-30,207.421268,58.093571,149.327697,17.892327,0.472223,40,...,0.00,0.0,0.75,1.0,-0.25,NaN,104,True,False,0.0
12924,trend_floor,252,2020-01-02,2020-12-30,207.421268,58.093571,149.327697,17.892327,0.472223,40,...,0.05,0.0,0.75,1.0,-0.25,0.15,104,True,False,0.0
12926,trend_floor,252,2020-01-02,2020-12-30,207.421268,58.093571,149.327697,17.892327,0.472223,40,...,0.05,0.0,0.75,1.0,-0.25,0.20,104,True,False,0.0


In [11]:
# Persist daily rows for the best lagged and best same-day variants for visual inspection.
selected_variants = []
for mode in ["lagged", "same_day"]:
    top = grid[grid["mode"].eq(mode)].sort_values(["objective_loss_20dd", "relative_return_pp"], ascending=[True, False]).iloc[0]
    cfg = {k: top[k] for k in [
        "min_target", "max_target", "max_base_position", "min_anchor_gap_180", "min_mom_63",
        "min_trend_slope_63", "min_trend_gap_63", "max_vix_percentile_252", "max_realized_vol_63",
        "min_drawdown_63", "max_equity_drawdown",
    ]}
    cfg = {k: (None if pd.isna(v) else float(v)) for k, v in cfg.items()}
    metrics, out = overlay_score(cfg, mode)
    out["variant_label"] = f"best_{mode}"
    selected_variants.append(out)

top_daily = pd.concat(selected_variants, ignore_index=True)
top_daily.to_csv(ARTIFACTS["trend_overlay_top_daily"], index=False)
top_daily.groupby("variant_label").agg(
    rows=("datetime", "size"),
    lifted_days=("trend_participation_lifted", "sum"),
    avg_pos=("position", "mean"),
    final_nav=("strategy_nav", "last"),
).reset_index()


,variant_label,rows,lifted_days,avg_pos,final_nav
0,best_lagged,252,73,0.430626,1.924894e+06
1,best_same_day,252,104,0.472223,3.074213e+06


In [12]:
# Feature-state coverage on top missed-upside days.
top_gap = gap_contribution.head(40).copy()
state_summary = []
for threshold in [0.10, 0.25, 0.40, 0.55]:
    mask = (
        (pd.to_numeric(top_gap["anchor_gap_180"], errors="coerce") >= threshold)
        & (pd.to_numeric(top_gap["mom_63"], errors="coerce") >= 0.20)
        & (pd.to_numeric(top_gap["vix_percentile_252"], errors="coerce") <= 0.75)
        & (pd.to_numeric(top_gap["drawdown_63"], errors="coerce") >= -0.25)
    )
    state_summary.append({
        "anchor_threshold": threshold,
        "top_gap_rows": int(len(top_gap)),
        "covered_rows": int(mask.sum()),
        "covered_share": float(mask.mean()),
        "covered_missed_upside_pp": float(top_gap.loc[mask, "legacy_default_missed_upside_pp"].sum()),
        "total_top_missed_upside_pp": float(top_gap["legacy_default_missed_upside_pp"].sum()),
    })
feature_state_summary = pd.DataFrame(state_summary)
feature_state_summary.to_csv(ARTIFACTS["feature_state_summary"], index=False)
feature_state_summary


,anchor_threshold,top_gap_rows,covered_rows,covered_share,covered_missed_upside_pp,total_top_missed_upside_pp
0,0.10,40,14,0.350,76.999282,324.411157
1,0.25,40,9,0.225,51.651447,324.411157
2,0.40,40,6,0.150,30.721097,324.411157
3,0.55,40,5,0.125,26.350740,324.411157


In [13]:
best_lagged = grid[grid["mode"].eq("lagged")].sort_values(["objective_loss_20dd", "relative_return_pp"], ascending=[True, False]).iloc[0].to_dict()
best_same = grid[grid["mode"].eq("same_day")].sort_values(["objective_loss_20dd", "relative_return_pp"], ascending=[True, False]).iloc[0].to_dict()
lagged_gate_count = int(grid[grid["mode"].eq("lagged")]["gate_pass_20dd"].sum())
same_day_gate_count = int(grid[grid["mode"].eq("same_day")]["gate_pass_20dd"].sum())

primary_blocker = "lagged_trend_participation_counterfactual_gap"
if lagged_gate_count > 0:
    primary_blocker = "official_replay_required_for_lagged_counterfactual"
elif same_day_gate_count > 0:
    primary_blocker = "same_day_only_counterfactual_leakage_risk"

decision = "release16_wf2020_trend_participation_gap_no_promotion"
selection = {
    "experiment": "release16_wf2020_trend_participation_gap_v1",
    "decision": decision,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "training_allowed_from_this_card": False,
    "primary_blocker": primary_blocker,
    "source_card": "release16_downtrend_drawdown_objective_screen_v1",
    "selected_release16_candidate": "wf_2020/dd_control_checkpoint",
    "best_lagged_counterfactual": best_lagged,
    "best_same_day_counterfactual": best_same,
    "lagged_gate_pass_count": lagged_gate_count,
    "same_day_gate_pass_count": same_day_gate_count,
    "target_relative_return_pp": 30.0,
    "target_max_dd_pct": 20.0,
    "next_mechanism": "convert_visible_wf2020_trend_participation_gap_to_fold_local_checkpoint_target_then_rerun_three_fold_official_replay",
    "sidecars": {k: str(v.relative_to(ROOT)) for k, v in ARTIFACTS.items()},
}

summary = {
    **selection,
    "release16_selected_metrics": selected_metrics,
    "comparison_metrics": comparison.to_dict("records"),
    "reconstruction_metrics": reconstruction.to_dict("records"),
    "top_gap_months_legacy": monthly_gap[monthly_gap["teacher"].eq("legacy_default")].head(6).to_dict("records"),
    "feature_state_summary": feature_state_summary.to_dict("records"),
    "summary_points": [
        f"Release16 wf_2020 selected path has relative return {selected_metrics['relative_return_pp']:.2f}pp and MaxDD {selected_metrics['max_dd_pct']:.2f}%.",
        f"SOXLBacktest legacy_default is the buy/sell timing reference but has wf_2020 MaxDD {comparison.loc[comparison['label'].eq('soxlbacktest_legacy_default'), 'max_dd_pct'].iloc[0]:.2f}% and is not promotable.",
        f"Trend-floor grid produced {lagged_gate_count} lagged and {same_day_gate_count} same-day counterfactual rows passing +30pp/<20% before official replay.",
        "No actor or runtime overlay is retained from this diagnostic; any useful row must become fold-local target/checkpoint scoring and pass three-fold official replay.",
    ],
}

ARTIFACTS["selection"].write_text(json.dumps(selection, indent=2, sort_keys=True), encoding="utf-8")
ARTIFACTS["summary"].write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
conclusion = "\n".join([
    "# release16_wf2020_trend_participation_gap_v1",
    "",
    f"- Decision: `{decision}`.",
    f"- Release16 `wf_2020/dd_control_checkpoint` remains below the relative-return floor: `{selected_metrics['relative_return_pp']:.2f}pp` vs target `+30pp`, while MaxDD is `{selected_metrics['max_dd_pct']:.2f}%` under the user `<20%` gate.",
    f"- Counterfactual trend-floor pass counts: lagged `{lagged_gate_count}`, same-day `{same_day_gate_count}`. Same-day rows are treated as leakage-risk diagnostics, not runnable evidence.",
    f"- Primary blocker: `{primary_blocker}`.",
    "- Next: convert the visible trend-participation gap to a fold-local checkpoint/target objective and rerun full three-fold official replay; do not promote this card.",
    "",
    "Artifacts:",
    *[f"- `{str(path.relative_to(ROOT))}`" for path in ARTIFACTS.values()],
])
ARTIFACTS["conclusion"].write_text(conclusion, encoding="utf-8")
summary

{'experiment': 'release16_wf2020_trend_participation_gap_v1',
 'decision': 'release16_wf2020_trend_participation_gap_no_promotion',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'training_allowed_from_this_card': False,
 'primary_blocker': 'official_replay_required_for_lagged_counterfactual',
 'source_card': 'release16_downtrend_drawdown_objective_screen_v1',
 'selected_release16_candidate': 'wf_2020/dd_control_checkpoint',
 'best_lagged_counterfactual': {'label': 'trend_floor',
  'rows': 252,
  'start': '2020-01-02',
  'end': '2020-12-30',
  'strategy_return_pct': 92.48938988895246,
  'buyhold_return_pct': 58.093570616483994,
  'relative_return_pp': 34.395819272468465,
  'max_dd_pct': 19.072100462081774,
  'avg_pos': 0.4306255896842804,
  'trade_count': 40,
  'mode': 'lagged',
  'lifted_days': 73,
  'avg_lift': 0.16944901140676072,
  'max_lift': 0.6543314651992784,
  'min_target': 0.75,
  'max_target': 0.95,
  'max_base_position': 0.3,
  'min_anchor_gap_180': 0.1,
 

## Conclusion

This diagnostic keeps the release16 drawdown repair intact but confirms that `wf_2020` still needs a trend-participation repair before any actor can be retained. The SOXLBacktest legacy path remains useful as buy/sell timing reference, not as a promotable model, because its drawdown is far above the `<20%` gate. Counterfactual trend floors in this notebook are diagnostic only: any row that appears useful must be converted into a fold-local target/checkpoint objective and rerun through all official walk-forward folds before retention or dashboard promotion.
